## 5 — Gráficos: Análise de Evasão e Retenção

| Código | Tipo | Descrição |
|--------|------|-----------|
|16 | Barras empilhadas | Motivos de saída por ano |
| 21 | Barras empilhadas | Conclusões no prazo vs. com atraso |
| * | Heatmap |    fazer análise de coorte por curso? |

### 5.1. Importações e Configurações

In [13]:
import pandas as pd
import numpy as np
import plotly.express as px

CORES_SITUACAO = {
    "Ingressante":          "#A5D6A7",
    "Em fluxo":             "#4CAF50",
    "Retido":               "#FF9800",
    "Concluída no prazo":   "#2196F3",
    "Concluída com atraso": "#64B5F6",
    "Abandono":             "#F44336",
    "Desligada":            "#E91E63",
    "Transf. externa":      "#9E9E9E",
    "Transf. interna":      "#BDBDBD",
    "Integralizada":        "#00BCD4",
}


### 5.2. Carga dos dados e cálculo dos indicadores

In [14]:
df_dados_tratados = pd.read_parquet("../dados_tratados/restinga_dados_tratados.parquet")

print("Shape:", df_dados_tratados.shape)
df_dados_tratados.head(3)


Shape: (10445, 19)


,Ano,Categoria da Situação,Cor / Raça,Código da Matricula,Código do Ciclo Matricula,Data de Fim Previsto do Ciclo,Data de Inicio do Ciclo,Data de Ocorrencia da Matricula,Eixo Tecnológico,Faixa Etária,Mês De Ocorrência da Situação,Nome de Curso,Renda Familiar,Sexo,Situação de Matrícula,Subeixo Tecnológico,Tipo de Curso,Turno,Concluinte_Previsto
0,2017,Em curso,Não declarada,44681986,1207788,2014-12-22,2012-03-05,2012-03-01,Informação e Comunicação,35 a 39 anos,2018-03-05,Análise e Desenvolvimento de Sistemas,Não declarada,F,Retido,Informática,Superior-Tecnologia,Matutino,False
1,2017,Em curso,Branca,66262145,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,2016-02-01,Técnico em Eletrônica,"0,5<RFP<=1",F,Em fluxo,Elétrica,Técnico-Integrado,Matutino,False
2,2017,Em curso,Branca,66262155,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,2016-02-01,Técnico em Eletrônica,"0,5<RFP<=1",F,Em fluxo,Elétrica,Técnico-Integrado,Matutino,False


In [15]:
df_dados_tratados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10445 entries, 0 to 10444
Data columns (total 19 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Ano                              10445 non-null  int64         
 1   Categoria da Situação            10445 non-null  object        
 2   Cor / Raça                       10445 non-null  object        
 3   Código da Matricula              10445 non-null  int64         
 4   Código do Ciclo Matricula        10445 non-null  int64         
 5   Data de Fim Previsto do Ciclo    10445 non-null  datetime64[ns]
 6   Data de Inicio do Ciclo          10445 non-null  datetime64[ns]
 7   Data de Ocorrencia da Matricula  10445 non-null  datetime64[ns]
 8   Eixo Tecnológico                 10445 non-null  object        
 9   Faixa Etária                     10445 non-null  object        
 10  Mês De Ocorrência da Situação    10445 non-null  datetime6

In [16]:
def calcular_indicadores(df, agrupamento):
    
    df_indicadores = (
        df.groupby(agrupamento)
        .agg(
            ingressantes    = ("Situação de Matrícula", lambda x: (x == "Ingressante").sum()),
            em_curso        = ("Categoria da Situação",  lambda x: (x == "Em curso").sum()),
            concluintes     = ("Categoria da Situação",  lambda x: (x == "Concluintes").sum()),
            evadidos        = ("Categoria da Situação",  lambda x: (x == "Evadidos").sum()),
            matr_atendidas  = ("Código da Matricula",    "count"),
            conc_no_prazo   = ("Situação de Matrícula",  lambda x: (x == "Concluída no prazo").sum()),
            conc_com_atraso = ("Situação de Matrícula",  lambda x: (x == "Concluída com atraso").sum()),
            abandono        = ("Situação de Matrícula",  lambda x: (x == "Abandono").sum()),
            desligados      = ("Situação de Matrícula",  lambda x: (x == "Desligada").sum()),
            transf_ext      = ("Situação de Matrícula",  lambda x: (x == "Transf. externa").sum()),
            transf_int      = ("Situação de Matrícula",  lambda x: (x == "Transf. interna").sum()),
            integralizadas  = ("Situação de Matrícula",  lambda x: (x == "Integralizada").sum()),
            conc_previstos  = ("Concluinte_Previsto",    "sum"),
            MREG            = ("Situação de Matrícula",
                               lambda x: ((x == "Em fluxo") | (x == "Ingressante")).sum()),
            MRET            = ("Situação de Matrícula",  lambda x: (x == "Retido").sum()),
        )
        .reset_index()
    )

    ma = df_indicadores["matr_atendidas"].replace(0, np.nan)

    df_indicadores["TC"] = (
        (df_indicadores["conc_no_prazo"] + df_indicadores["conc_com_atraso"]) / ma * 100
    )
    df_indicadores["TE"] = (
        (df_indicadores["abandono"] + df_indicadores["desligados"] + df_indicadores["transf_ext"])
        / ma * 100
    )
    df_indicadores["TR"] = df_indicadores["MRET"] / ma * 100

    matr_finalizadas = (
        df_indicadores["conc_no_prazo"] + df_indicadores["conc_com_atraso"]
        + df_indicadores["abandono"] + df_indicadores["desligados"]
        + df_indicadores["transf_int"] + df_indicadores["transf_ext"]
    ).replace(0, np.nan)
    df_indicadores["IEf"] = (
        (df_indicadores["conc_no_prazo"] + df_indicadores["conc_com_atraso"])
        / matr_finalizadas * 100
    )

     # TMREG — Taxa de Matrícula Continuada Regular
    df_indicadores["TMREG"] = df_indicadores["MREG"] / ma * 100

    # TPE — Taxa de Permanência e Êxito
    df_indicadores["TPE"] = df_indicadores["TC"] + df_indicadores["TMREG"]
    return df_indicadores.fillna(0).round(2)


# Calcula para diferentes granularidades
ind_ano_curso = calcular_indicadores(df_dados_tratados, ["Ano", "Nome de Curso"])
ind_ano_tipo  = calcular_indicadores(df_dados_tratados, ["Ano", "Tipo de Curso"])

# Identifica o último ano disponível
ultimo_ano = int(ind_ano_curso["Ano"].max())

print("ind_ano_curso:", ind_ano_curso.shape)
print("Último ano disponível:", ultimo_ano)

ind_ano_curso: (86, 23)
Último ano disponível: 2024


### 5.3. Gráficos

In [17]:
# 16: Motivos de saída por ano (barras empilhadas)

situacoes_saida = ["Abandono", "Desligada", "Transf. externa", "Transf. interna", "Reprovada"]

df_saidas = (
    df_dados_tratados[df_dados_tratados["Situação de Matrícula"].isin(situacoes_saida)]
    .groupby(["Ano", "Situação de Matrícula"])["Código da Matricula"]
    .count()
    .reset_index(name="Qtd")
)

fig_g12 = px.bar(
    df_saidas,
    x="Ano",
    y="Qtd",
    color="Situação de Matrícula",
    barmode="stack",
    color_discrete_map=CORES_SITUACAO,
    title="Motivos de Saída por Ano",
    labels={"Qtd": "Matrículas", "Situação de Matrícula": "Motivo"},
    text_auto=True,
)
fig_g12.update_xaxes(tickmode="linear", dtick=1)
fig_g12.show()

In [18]:
# : Conclusões no prazo vs. com atraso por curso

conc_prazo = (
    df_dados_tratados[df_dados_tratados["Categoria da Situação"] == "Concluintes"]
    .groupby(["Nome de Curso", "Situação de Matrícula"])["Código da Matricula"]
    .count()
    .reset_index(name="Qtd")
)

fig_g17 = px.bar(
    conc_prazo,
    x="Nome de Curso",
    y="Qtd",
    color="Situação de Matrícula",
    barmode="stack",
    color_discrete_map=CORES_SITUACAO,
    title="Conclusões: No Prazo vs. Com Atraso por Curso",
    labels={"Qtd": "Concluintes", "Nome de Curso": ""},
    text_auto=True,
)
fig_g17.update_layout(xaxis_tickangle=-30)
fig_g17.show()